
# ProbeSelector 實驗筆記
以開源 tabular datasets（Kaggle Otto Group Product Classification + Superconductivity Data）示範自製 `ProbeSelector`，展示 SHAP 重要度模式、字串化 CV 參數，並使用 LightGBM / CatBoost 這類真實專案常見的模型。



## 1. 載入資料與簡介
- **分類**：Otto Group Product Classification（61,878 筆 × 93 features，OpenML #41082）。示範時抽樣 10,000 筆，以保留高維度又兼顧可操作性。
- **回歸**：Superconductivity Data（21,263 筆 × 81 features，OpenML #422），同樣抽樣 10,000 筆，目標欄位為 `critical_temp`。
- 兩個 dataset 都大於傳統 toy data，因此更能驗證 ProbeSelector 在實務情境下的表現。


In [1]:

import numpy as np
import pandas as pd

from sklearn.datasets import fetch_openml
from sklearn.model_selection import StratifiedKFold, KFold, cross_val_score

from lightgbm import LGBMClassifier
from catboost import CatBoostRegressor


In [2]:

OTTO_SAMPLE_SIZE = 10_000

otto = fetch_openml('otto-group-product-classification-challenge', version=1, as_frame=True)
otto_df = otto.frame.sample(n=OTTO_SAMPLE_SIZE, random_state=42)

X_cls = otto_df.drop(columns=['target']).astype(np.float32)
y_cls = otto_df['target'].astype('category').cat.codes
cls_full_df = X_cls.assign(target=y_cls)

print(f"Otto subset — Samples: {X_cls.shape[0]:,} | Features: {X_cls.shape[1]} | Classes: {y_cls.nunique()}")
X_cls.head()


Otto subset — Samples: 10,000 | Features: 93 | Classes: 9


,feat_1,feat_2,feat_3,feat_4,feat_5,feat_6,feat_7,feat_8,feat_9,feat_10,...,feat_84,feat_85,feat_86,feat_87,feat_88,feat_89,feat_90,feat_91,feat_92,feat_93
47416,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
46490,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,...,0.0,0.0,8.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
35460,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
44640,9.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,4.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
32087,0.0,2.0,9.0,4.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0



## 2. 建立分類 baseline
使用 `LGBMClassifier` + Stratified 5-fold (`roc_auc_ovr_weighted`) 當作 baseline，稍後與 ProbeSelector 結果比較。


In [ ]:

clf = LGBMClassifier(
    n_estimators=600,
    learning_rate=0.05,
    num_leaves=64,
    max_depth=5,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42,
    verbose=-1
)
cls_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cls_scoring = 'roc_auc_ovr_weighted'

cls_baseline_scores = cross_val_score(clf, X_cls, y_cls, cv=cls_cv, scoring=cls_scoring)
print(f"Baseline ROC-AUC (weighted OVR): {cls_baseline_scores.mean():.4f} ± {cls_baseline_scores.std():.4f}")

Baseline ROC-AUC (weighted OVR): 0.9629 ± 0.0013


: 


## 3. ProbeSelector 主要參數
| 參數 | Demo 設定 | 備註 |
| --- | --- | --- |
| `estimator` | `LGBMClassifier(...)` / `CatBoostRegressor(...)` | 皆能與 probe 一起訓練，並支援 SHAP 或 feature_importances_。 |
| `importance_mode` | `'shap'`（分類） / `'collective'`（回歸） | `'collective'` 讀 feature_importances_ / coef，`'individual'` 逐欄 CV，`'shap'` 透過 SHAP values。 |
| `cv` | `'stratified'` / `'kfold'` | 以字串指定交叉驗證類型，並搭配 `cv_splits=5` 自動建立物件；`'auto'` 則依目標型態判斷。 |
| `n_probes` | 8（分類） / 10（回歸） | Probe 為 noise baseline，提供 mean/median/quantile 門檻參考。 |
| `threshold_strategy` | `'median'` / `'quantile'` | 搭配 probe 統計決定門檻；SHAP distribution 偏態時使用 median/quantile 較穩定。 |
| `batch_remove` | `'percentile'` / `'all_below'` | 控制每輪刪除幅度，避免一次移除過多特徵。 |
| `keep_min_features` | 20（分類） / 25（回歸） | 保底避免模型無特徵可用。 |
| `shap_max_samples` | 2048 | 設定 SHAP 計算上限樣本數，以免在大型資料集耗盡記憶體。 |
| `verbose` + `get_iteration_logs()` | `True` | 逐輪輸出門檻、刪除名單與 probe 統計，方便除錯。 |

> `collective` 參數仍保留相容性，但建議透過 `importance_mode` 來描述想要的行為。



### 3.1 推薦預設模型
- **分類**：`LGBMClassifier`、`CatBoostClassifier` 皆具備 tree-based importance，且 SHAP TreeExplainer 支援度高。
- **回歸**：`CatBoostRegressor`、`RandomForestRegressor` 對非線性特徵與 outlier 較有韌性，同時可搭配 collective/SHAP 模式。
- **僅能使用其他 estimator 時**：改用 `importance_mode='individual'` + 合適的 `scoring`；或先訓練可產生 importance 的代理模型，再用 `transform` 應用於原模型。



### 3.2 自訂 ProbeSelector 程式碼
下方為最新版 `ProbeSelector` 實作，已加入 SHAP 重要度、字串化 CV 設定與 iteration log。


In [6]:

from __future__ import annotations
from typing import Optional, List, Union, Literal, Dict, Any, Tuple

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import KFold, GroupKFold, StratifiedKFold, cross_validate
from sklearn.utils.validation import check_is_fitted

try:
    import shap
except ImportError:  # pragma: no cover
    shap = None

try:  # LightGBM for自動估計器
    from lightgbm import LGBMClassifier, LGBMRegressor
except ImportError:  # pragma: no cover
    LGBMClassifier = None
    LGBMRegressor = None

try:  # CatBoost for自動估計器
    from catboost import CatBoostClassifier, CatBoostRegressor
except ImportError:  # pragma: no cover
    CatBoostClassifier = None
    CatBoostRegressor = None

ThresholdStrategy = Literal['mean', 'median', 'min', 'max', 'quantile']
ImportanceMode = Literal['collective', 'individual', 'shap']
CVStrategy = Literal['auto', 'stratified', 'kfold', 'group']


class ProbeSelector:
    """
    Probe Feature Selection 增強版：支援 SHAP 重要度、逐欄 CV、迭代刪除與完整 trace/log。

    Parameters
    ----------
    estimator : Optional[sklearn-like estimator]
        可不預先提供，若為 None 將依目標型態自動挑選 (LightGBM / CatBoost / RandomForest)。
    y_col : str
        目標欄位名稱。
    feature_cols : Optional[List[str]]
        特徵欄位，`None` 代表自動使用除了 y 以外的所有欄位。
    collective : bool
        舊參數，保留相容性；若未指定 `importance_mode`，會落在 collective/individual 之間。
    importance_mode : {'collective','individual','shap'}
        collective：讀取 feature_importances_/coef；individual：逐欄 CV；shap：以 SHAP values 為重要度。
    scoring : str
        individual 模式需要的 sklearn scorer 名稱。
    n_probes : int
        產生的 probe 數量。
    distribution : {'normal','uniform','binary','all'}
        probe 分佈，`'all'` 會輪流三種。
    cv : {'auto','stratified','kfold','group'}
        以字串描述要建立的 CV 類型。
    cv_splits : int
        KFold / StratifiedKFold / GroupKFold 的切分數量。
    groups : Optional[np.ndarray]
        GroupKFold 需要的群組向量。
    random_state : int
        隨機種子。
    threshold_strategy : ThresholdStrategy
        probe 門檻策略。
    threshold_quantile : float
        當 strategy='quantile' 時的分位數。
    iterative : bool
        是否啟用迭代式刪除。
    max_iter : int
        迭代上限。
    batch_remove : {'all_below','percentile'}
        每輪刪除策略。
    bottom_percent : float
        當 batch_remove='percentile' 時要刪除的比例。
    keep_min_features : int
        至少保留的特徵數。
    shap_max_samples : Optional[int]
        SHAP 重要度最多使用多少樣本；`None` 代表使用全部樣本。
    verbose : bool
        True 時列印每輪 iteration summary 並保存在 `iteration_logs_`。
    """

    def __init__(
        self,
        estimator=None,
        y_col: str = 'target',
        feature_cols: Optional[List[str]] = None,
        *,
        collective: bool = True,
        importance_mode: ImportanceMode | None = None,
        scoring: str = 'r2',
        n_probes: int = 5,
        distribution: Literal['normal', 'uniform', 'binary', 'all'] = 'all',
        cv: CVStrategy = 'auto',
        cv_splits: int = 5,
        groups: Optional[np.ndarray] = None,
        random_state: int = 42,
        threshold_strategy: ThresholdStrategy = 'mean',
        threshold_quantile: float = 0.5,
        iterative: bool = False,
        max_iter: int = 5,
        batch_remove: Literal['all_below', 'percentile'] = 'all_below',
        bottom_percent: float = 0.1,
        keep_min_features: int = 1,
        shap_max_samples: Optional[int] = 2048,
        verbose: bool = False,
    ):
        self.estimator = estimator
        self.y_col = y_col
        self.feature_cols = feature_cols

        self.collective = collective
        if importance_mode is None:
            self.importance_mode = 'collective' if collective else 'individual'
        else:
            self.importance_mode = importance_mode
        self.scoring = scoring
        self.n_probes = n_probes
        self.distribution = distribution
        self.cv_strategy = cv
        self.cv_splits = int(cv_splits)
        self.groups = groups
        self.random_state = random_state

        self.threshold_strategy = threshold_strategy
        self.threshold_quantile = float(threshold_quantile)

        self.iterative = iterative
        self.max_iter = max_iter
        self.batch_remove = batch_remove
        self.bottom_percent = float(bottom_percent)
        self.keep_min_features = keep_min_features
        self.shap_max_samples = shap_max_samples
        self.verbose = verbose

        self.selected_features_: List[str] = []
        self.features_to_drop_: List[str] = []
        self.importance_table_: pd.DataFrame = pd.DataFrame()
        self.decision_trace_: pd.DataFrame = pd.DataFrame()
        self.probe_columns_: List[str] = []
        self.last_threshold_: float = np.nan
        self.last_threshold_detail_: Dict[str, Any] = {}
        self.iteration_logs_: List[Dict[str, Any]] = []
        self.auto_estimator_metadata_: Dict[str, Any] = {}
        self._estimator_template_ = estimator

    @staticmethod
    def _sample(kind: str, n: int, rng) -> np.ndarray:
        if kind == 'normal':
            return rng.standard_normal(n)
        if kind == 'uniform':
            return rng.uniform(size=n)
        if kind == 'binary':
            return rng.integers(0, 2, size=n)
        return rng.standard_normal(n)

    def _make_probes(self, n_rows: int, rng) -> pd.DataFrame:
        probes: Dict[str, np.ndarray] = {}
        if self.distribution == 'all':
            cycle = ['normal', 'uniform', 'binary']
            for i in range(self.n_probes):
                d = cycle[i % 3]
                probes[f'__probe_{d}_{i}'] = self._sample(d, n_rows, rng)
        else:
            for i in range(self.n_probes):
                probes[f'__probe_{self.distribution}_{i}'] = self._sample(self.distribution, n_rows, rng)
        return pd.DataFrame(probes)

    def _build_cv(self, y: np.ndarray):
        strategy = self.cv_strategy
        if strategy == 'group':
            if self.groups is None:
                raise ValueError('cv="group" 需要提供 groups。')
            return GroupKFold(n_splits=self.cv_splits)
        if strategy == 'stratified':
            return StratifiedKFold(n_splits=self.cv_splits, shuffle=True, random_state=self.random_state)
        if strategy == 'kfold':
            return KFold(n_splits=self.cv_splits, shuffle=True, random_state=self.random_state)
        if strategy == 'auto':
            is_classification = self._is_classification_task(y)
            if is_classification:
                return StratifiedKFold(n_splits=self.cv_splits, shuffle=True, random_state=self.random_state)
            return KFold(n_splits=self.cv_splits, shuffle=True, random_state=self.random_state)
        raise ValueError(f'Unknown cv strategy: {strategy}')

    def _threshold_from(self, values: np.ndarray) -> float:
        if self.threshold_strategy == 'mean':
            thr = float(np.mean(values))
        elif self.threshold_strategy == 'median':
            thr = float(np.median(values))
        elif self.threshold_strategy == 'min':
            thr = float(np.min(values))
        elif self.threshold_strategy == 'max':
            thr = float(np.max(values))
        elif self.threshold_strategy == 'quantile':
            q = np.clip(self.threshold_quantile, 0.0, 1.0)
            thr = float(np.quantile(values, q))
        else:
            raise ValueError(f'Unknown threshold_strategy: {self.threshold_strategy}')
        self.last_threshold_detail_ = {
            'strategy': self.threshold_strategy,
            'quantile': self.threshold_quantile if self.threshold_strategy == 'quantile' else None,
            'probe_stats': {
                'mean': float(np.mean(values)),
                'median': float(np.median(values)),
                'min': float(np.min(values)),
                'max': float(np.max(values)),
                'std': float(np.std(values)),
            },
        }
        return thr

    def _clone_base_estimator(self):
        if self._estimator_template_ is None:
            raise RuntimeError('Estimator 尚未初始化，請先呼叫 fit。')
        return clone(self._estimator_template_)

    def _compute_shap_importance(
        self,
        estimator,
        evaluation_data: pd.DataFrame,
        background_data: Optional[pd.DataFrame] = None,
        *,
        seed_offset: int = 0,
    ) -> pd.Series:
        if shap is None:
            raise ImportError('shap 未安裝，請先 pip install shap。')
        if evaluation_data.empty:
            return pd.Series(0.0, index=evaluation_data.columns, dtype=float)
        if background_data is None:
            background_data = evaluation_data
        bg = background_data
        eval_df = evaluation_data
        sample_seed = None if self.random_state is None else self.random_state + seed_offset
        if self.shap_max_samples is not None and len(bg) > self.shap_max_samples:
            bg = bg.sample(self.shap_max_samples, random_state=sample_seed)
        if self.shap_max_samples is not None and len(eval_df) > self.shap_max_samples:
            eval_df = eval_df.sample(self.shap_max_samples, random_state=sample_seed)
        explainer = shap.Explainer(estimator, bg)
        explanation = explainer(eval_df)
        values = getattr(explanation, 'values', explanation)
        values = np.asarray(values)
        if values.ndim == 1:
            values = values.reshape(-1, 1)
        if values.ndim == 3:
            if values.shape[2] == eval_df.shape[1]:
                values = values.mean(axis=1)
            elif values.shape[1] == eval_df.shape[1]:
                values = values.mean(axis=2)
            else:
                values = values.reshape(values.shape[0], -1, eval_df.shape[1]).mean(axis=1)
        elif values.ndim > 3:
            values = values.reshape(values.shape[0], -1, eval_df.shape[1]).mean(axis=1)
        abs_vals = np.abs(values)
        importances = abs_vals.mean(axis=0)
        series = pd.Series(importances, index=eval_df.columns, dtype=float)
        return series.reindex(evaluation_data.columns)

    def _single_round(self, X: pd.DataFrame, y: np.ndarray, features: List[str], iteration: int) -> Dict[str, Any]:
        rng = np.random.default_rng(self.random_state + iteration)
        probes = self._make_probes(len(X), rng)
        probes.index = X.index
        X_plus = pd.concat([X[features], probes], axis=1)
        probe_cols = list(probes.columns)

        mode = self.importance_mode
        if mode == 'collective':
            est = self._clone_base_estimator()
            est.fit(X_plus, y)
            if hasattr(est, 'feature_importances_'):
                imp = np.asarray(est.feature_importances_, dtype=float)
            elif hasattr(est, 'coef_'):
                coef = est.coef_
                imp = np.abs(coef if getattr(coef, 'ndim', 1) == 1 else coef.ravel())
            else:
                raise ValueError('collective 模式需要 estimator 具備 feature_importances_ 或 coef_.')
            importances = pd.Series(imp, index=X_plus.columns)
        elif mode == 'shap':
            cv_obj = self._build_cv(y)
            fold_importances: List[pd.Series] = []
            for fold_idx, (train_idx, val_idx) in enumerate(cv_obj.split(X_plus, y, groups=self.groups)):
                est = self._clone_base_estimator()
                X_train = X_plus.iloc[train_idx]
                X_val = X_plus.iloc[val_idx]
                y_train = y[train_idx]
                est.fit(X_train, y_train)
                fold_imp = self._compute_shap_importance(
                    est,
                    X_val,
                    background_data=X_train,
                    seed_offset=fold_idx,
                )
                fold_importances.append(fold_imp)
            if fold_importances:
                importances = pd.concat(fold_importances, axis=1).mean(axis=1)
            else:
                est = self._clone_base_estimator()
                est.fit(X_plus, y)
                importances = self._compute_shap_importance(est, X_plus)
            importances = importances.reindex(X_plus.columns)
        elif mode == 'individual':
            from sklearn.metrics import get_scorer
            scorer = get_scorer(self.scoring)
            cv_obj = self._build_cv(y)
            scores: Dict[str, float] = {}
            for col in X_plus.columns:
                est = self._clone_base_estimator()
                cv_res = cross_validate(
                    est,
                    X_plus[[col]],
                    y,
                    scoring=scorer,
                    cv=cv_obj,
                    groups=self.groups,
                    return_train_score=False,
                )
                scores[col] = float(np.mean(cv_res['test_score']))
            importances = pd.Series(scores, index=X_plus.columns)
        else:
            raise ValueError(f'Unknown importance_mode: {mode}')

        probe_vals = importances.loc[probe_cols].values
        thr = self._threshold_from(probe_vals)

        real_imp = importances.loc[features]
        below_mask = real_imp <= thr

        if self.batch_remove == 'all_below':
            to_remove = list(real_imp.index[below_mask])
        elif self.batch_remove == 'percentile':
            k = max(1, int(np.ceil(len(real_imp) * self.bottom_percent)))
            to_remove = list(real_imp.sort_values(ascending=True).index[:k])
        else:
            raise ValueError(f'Unknown batch_remove: {self.batch_remove}')

        if len(features) - len(to_remove) < self.keep_min_features:
            allowed = real_imp.sort_values(ascending=False).index[: self.keep_min_features]
            to_remove = list(set(features) - set(allowed))

        decisions = []
        for f in features:
            imp_val = float(real_imp.get(f, np.nan))
            if f in to_remove:
                reason = 'below_threshold' if self.batch_remove == 'all_below' else 'bottom_percentile'
                decisions.append({
                    'iteration': iteration,
                    'feature': f,
                    'importance': imp_val,
                    'threshold': float(thr),
                    'decision': 'remove',
                    'reason': reason,
                    **self.last_threshold_detail_['probe_stats'],
                })
            else:
                decisions.append({
                    'iteration': iteration,
                    'feature': f,
                    'importance': imp_val,
                    'threshold': float(thr),
                    'decision': 'keep',
                    'reason': 'above_threshold',
                    **self.last_threshold_detail_['probe_stats'],
                })

        log_entry = {
            'iteration': iteration,
            'importance_mode': mode,
            'n_features_start': len(features),
            'threshold': float(thr),
            'n_removed': len(to_remove),
            'removed': list(to_remove),
            'n_features_after': len(features) - len(to_remove),
            'strategy': self.threshold_strategy,
        }
        log_entry.update({f'probe_{k}': v for k, v in self.last_threshold_detail_['probe_stats'].items()})
        if self.verbose:
            print(
                f"[ProbeSelector] Iter {iteration}: {len(features)} -> {log_entry['n_features_after']} features "
                f"(removed {len(to_remove)}) | thr={thr:.4f} | mode={mode}"
            )

        return dict(
            importances=importances.sort_values(ascending=False),
            probe_cols=probe_cols,
            threshold=float(thr),
            decisions=pd.DataFrame(decisions),
            to_remove=to_remove,
            log_entry=log_entry,
        )

    def fit(self, df: pd.DataFrame):
        if self.y_col not in df.columns:
            raise ValueError(f"y_col='{self.y_col}' 不在 DataFrame 欄位內。")
        y = df[self.y_col].to_numpy()
        if self.feature_cols is None:
            features = [c for c in df.columns if c != self.y_col]
        else:
            features = list(self.feature_cols)

        X = df[features].copy()

        base_estimator = self.estimator
        auto_meta: Dict[str, Any] = {}
        if base_estimator is None:
            base_estimator, auto_meta = self._auto_estimator(X, y)
        self.estimator = base_estimator
        self._estimator_template_ = base_estimator
        self.auto_estimator_metadata_ = auto_meta

        all_decisions = []
        final_importances = None
        final_probes: List[str] = []
        last_thr = np.nan
        self.iteration_logs_ = []

        if not self.iterative:
            round_out = self._single_round(X, y, features, iteration=1)
            final_importances = round_out['importances']
            final_probes = round_out['probe_cols']
            last_thr = round_out['threshold']
            all_decisions.append(round_out['decisions'])
            self.iteration_logs_.append(round_out['log_entry'])
            to_remove = set(round_out['to_remove'])
            self.selected_features_ = [f for f in features if f not in to_remove]
            self.features_to_drop_ = [f for f in features if f in to_remove]
        else:
            cur_features = features[:]
            for it in range(1, self.max_iter + 1):
                round_out = self._single_round(X, y, cur_features, iteration=it)
                all_decisions.append(round_out['decisions'])
                self.iteration_logs_.append(round_out['log_entry'])
                last_thr = round_out['threshold']
                final_importances = round_out['importances']
                final_probes = round_out['probe_cols']
                to_remove = round_out['to_remove']
                if len(to_remove) == 0:
                    break
                next_features = [f for f in cur_features if f not in to_remove]
                if len(next_features) >= len(cur_features):
                    break
                cur_features = next_features
                if len(cur_features) <= self.keep_min_features:
                    break
            self.selected_features_ = cur_features
            self.features_to_drop_ = [f for f in features if f not in cur_features]

        self.importance_table_ = pd.DataFrame(
            {
                'feature': final_importances.index,
                'importance': final_importances.values,
                'is_probe': [f in final_probes for f in final_importances.index],
            }
        ).reset_index(drop=True)
        self.decision_trace_ = pd.concat(all_decisions, axis=0, ignore_index=True)
        self.probe_columns_ = final_probes
        self.last_threshold_ = float(last_thr)
        return self

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        check_is_fitted(self, ['selected_features_'])
        return df[self.selected_features_].copy()

    def fit_transform(self, df: pd.DataFrame) -> pd.DataFrame:
        return self.fit(df).transform(df)

    def __call__(self, df: pd.DataFrame) -> Tuple[List[str], List[str], pd.DataFrame]:
        """讓實例可被直接當作 callable：傳入 DataFrame，立即執行 fit 並回傳 (保留欄位, 移除欄位, 決策 trace)。"""
        fitted = self.fit(df)
        selected = list(fitted.selected_features_)
        dropped = list(fitted.features_to_drop_)
        trace = fitted.get_trace()
        return selected, dropped, trace

    def get_trace(self) -> pd.DataFrame:
        check_is_fitted(self, ['decision_trace_'])
        return self.decision_trace_.copy()

    def get_importances(self) -> pd.DataFrame:
        check_is_fitted(self, ['importance_table_'])
        return self.importance_table_.copy()

    def get_iteration_logs(self) -> pd.DataFrame:
        check_is_fitted(self, ['iteration_logs_'])
        return pd.DataFrame(self.iteration_logs_)

    def _is_classification_task(self, y: np.ndarray) -> bool:
        series = pd.Series(y)
        unique = series.nunique(dropna=False)
        if series.dtype.kind in {'O'}:
            return True
        if series.dtype.name == 'category':
            return True
        if series.dtype.kind in {'b', 'i', 'u'}:
            return unique <= max(20, series.size // 5)
        if series.dtype.kind == 'f':
            is_int_like = np.allclose(series, np.round(series))
            if is_int_like and unique <= 20:
                return True
        return False

    def _auto_estimator(self, X: pd.DataFrame, y: np.ndarray) -> Tuple[Any, Dict[str, Any]]:
        is_classification = self._is_classification_task(y)
        meta: Dict[str, Any] = {'auto': True}
        if is_classification:
            if LGBMClassifier is not None:
                meta['chosen'] = 'LGBMClassifier'
                return (
                    LGBMClassifier(
                        n_estimators=400,
                        learning_rate=0.05,
                        num_leaves=64,
                        subsample=0.9,
                        colsample_bytree=0.9,
                        random_state=self.random_state,
                    ),
                    meta,
                )
            meta['chosen'] = 'RandomForestClassifier'
            return (
                RandomForestClassifier(
                    n_estimators=300,
                    random_state=self.random_state,
                    n_jobs=-1,
                ),
                meta,
            )
        # regression fallback
        if CatBoostRegressor is not None:
            meta['chosen'] = 'CatBoostRegressor'
            return (
                CatBoostRegressor(
                    depth=8,
                    learning_rate=0.05,
                    iterations=600,
                    loss_function='RMSE',
                    random_state=self.random_state,
                    verbose=0,
                    allow_writing_files=False,
                ),
                meta,
            )
        meta['chosen'] = 'RandomForestRegressor'
        return (
            RandomForestRegressor(
                n_estimators=400,
                random_state=self.random_state,
                n_jobs=-1,
            ),
            meta,
        )



## 4. 建立並訓練 ProbeSelector（分類 + SHAP）
以 `importance_mode='shap'` 結合 probe baseline，觀察每輪門檻與被刪除的欄位；`cv='stratified'` 則確保 individual 模式也能用同樣 API。


In [10]:

selector = ProbeSelector(
    estimator=clf,
    y_col='target',
    feature_cols=None,
    collective=True,
    importance_mode='collective',
    scoring=cls_scoring,
    n_probes=8,
    distribution='all',
    cv='stratified',
    cv_splits=5,
    groups=None,
    random_state=42,
    threshold_strategy='median',
    iterative=True,
    max_iter=6,
    batch_remove='percentile',
    bottom_percent=0.15,
    keep_min_features=20,
    shap_max_samples=2048,
    verbose=True,
)

selector.fit(cls_full_df)

print(f"Original features: {X_cls.shape[1]}")
print(f"Selected features: {len(selector.selected_features_)}")
print(f"Probe columns: {len(selector.probe_columns_)} synthetic features")
print(f"Last probe threshold: {selector.last_threshold_:.4f}")
print(f"Dropped features ({len(selector.features_to_drop_)}): {selector.features_to_drop_[:10]}{' ...' if len(selector.features_to_drop_) > 10 else ''}")


[ProbeSelector] Iter 1: 93 -> 79 features (removed 14) | thr=5039.0000 | mode=collective
[ProbeSelector] Iter 2: 79 -> 67 features (removed 12) | thr=5383.0000 | mode=collective
[ProbeSelector] Iter 3: 67 -> 56 features (removed 11) | thr=5621.0000 | mode=collective
[ProbeSelector] Iter 4: 56 -> 47 features (removed 9) | thr=5819.0000 | mode=collective
[ProbeSelector] Iter 5: 47 -> 39 features (removed 8) | thr=5940.0000 | mode=collective
[ProbeSelector] Iter 6: 39 -> 33 features (removed 6) | thr=6450.0000 | mode=collective
Original features: 93
Selected features: 33
Probe columns: 8 synthetic features
Last probe threshold: 6450.0000
Dropped features (60): ['feat_1', 'feat_2', 'feat_3', 'feat_4', 'feat_5', 'feat_6', 'feat_7', 'feat_10', 'feat_12', 'feat_13'] ...


In [11]:

from IPython.display import display

importance_table = selector.get_importances()
probe_threshold = selector.last_threshold_
probe_stats = importance_table.loc[importance_table['is_probe'], 'importance'].describe()
iteration_logs = selector.get_iteration_logs()

real_features = (
    importance_table.loc[~importance_table['is_probe']]
    .assign(
        threshold=probe_threshold,
        margin=lambda df: df['importance'] - df['threshold'],
        dropped=lambda df: df['feature'].isin(selector.features_to_drop_),
    )
    .sort_values('importance', ascending=False)
)

print('Probe importance describe:')
print(probe_stats)

display(real_features.head(10))
display(real_features.query('dropped').sort_values('margin'))
display(selector.get_trace().head(10))
display(iteration_logs)


Probe importance describe:
count       8.000000
mean     5072.750000
std      2793.794795
min       488.000000
25%      4811.750000
50%      6450.000000
75%      6754.250000
max      6842.000000
Name: importance, dtype: float64


,feature,importance,is_probe,threshold,margin,dropped
6,feat_67,3593.0,False,6450.0,-2857.0,False
7,feat_24,3261.0,False,6450.0,-3189.0,False
8,feat_34,3176.0,False,6450.0,-3274.0,False
9,feat_40,2831.0,False,6450.0,-3619.0,False
10,feat_14,2621.0,False,6450.0,-3829.0,False
11,feat_25,2620.0,False,6450.0,-3830.0,False
12,feat_86,2451.0,False,6450.0,-3999.0,False
13,feat_62,2402.0,False,6450.0,-4048.0,False
14,feat_48,2379.0,False,6450.0,-4071.0,False
15,feat_15,2322.0,False,6450.0,-4128.0,False


,feature,importance,is_probe,threshold,margin,dropped
44,feat_47,1081.0,False,6450.0,-5369.0,True
43,feat_56,1127.0,False,6450.0,-5323.0,True
42,feat_71,1193.0,False,6450.0,-5257.0,True
41,feat_26,1266.0,False,6450.0,-5184.0,True
40,feat_33,1270.0,False,6450.0,-5180.0,True
39,feat_20,1304.0,False,6450.0,-5146.0,True


,iteration,feature,importance,threshold,decision,reason,mean,median,min,max,std
0,1,feat_1,774.0,5039.0,keep,above_threshold,3978.75,5039.0,440.0,5398.0,2039.001579
1,1,feat_2,268.0,5039.0,remove,bottom_percentile,3978.75,5039.0,440.0,5398.0,2039.001579
2,1,feat_3,319.0,5039.0,keep,above_threshold,3978.75,5039.0,440.0,5398.0,2039.001579
3,1,feat_4,387.0,5039.0,keep,above_threshold,3978.75,5039.0,440.0,5398.0,2039.001579
4,1,feat_5,220.0,5039.0,remove,bottom_percentile,3978.75,5039.0,440.0,5398.0,2039.001579
5,1,feat_6,215.0,5039.0,remove,bottom_percentile,3978.75,5039.0,440.0,5398.0,2039.001579
6,1,feat_7,367.0,5039.0,keep,above_threshold,3978.75,5039.0,440.0,5398.0,2039.001579
7,1,feat_8,1089.0,5039.0,keep,above_threshold,3978.75,5039.0,440.0,5398.0,2039.001579
8,1,feat_9,1472.0,5039.0,keep,above_threshold,3978.75,5039.0,440.0,5398.0,2039.001579
9,1,feat_10,424.0,5039.0,keep,above_threshold,3978.75,5039.0,440.0,5398.0,2039.001579


,iteration,importance_mode,n_features_start,threshold,n_removed,removed,n_features_after,strategy,probe_mean,probe_median,probe_min,probe_max,probe_std
0,1,collective,93,5039.0,14,"[feat_61, feat_82, feat_51, feat_6, feat_12, f...",79,median,3978.750,5039.0,440.0,5398.0,2039.001579
1,2,collective,79,5383.0,12,"[feat_27, feat_49, feat_63, feat_77, feat_45, ...",67,median,4165.500,5383.0,422.0,5740.0,2165.172106
2,3,collective,67,5621.0,11,"[feat_80, feat_89, feat_74, feat_4, feat_91, f...",56,median,4416.250,5621.0,470.0,6049.0,2271.934459
3,4,collective,56,5819.0,9,"[feat_30, feat_55, feat_44, feat_78, feat_18, ...",47,median,4519.375,5819.0,432.0,6065.0,2341.754254
4,5,collective,47,5940.0,8,"[feat_87, feat_92, feat_50, feat_83, feat_1, f...",39,median,4643.250,5940.0,425.0,6567.0,2440.196188
5,6,collective,39,6450.0,6,"[feat_47, feat_56, feat_71, feat_26, feat_33, ...",33,median,5072.750,6450.0,488.0,6842.0,2613.355733



## 5. 分類：重新評估
使用 `selector.transform` 過濾掉 probe baseline 以下的欄位，再以同樣的 Stratified 5-fold ROC-AUC（weighted OVR）檢視性能差異。


In [12]:

cls_selected = selector.transform(cls_full_df)
cls_selected_scores = cross_val_score(clf, cls_selected, y_cls, cv=cls_cv, scoring=cls_scoring)
print(
    f"After ProbeSelector ROC-AUC (weighted OVR): {cls_selected_scores.mean():.4f} ± {cls_selected_scores.std():.4f}"
)
print(
    f"ΔROC-AUC: {cls_selected_scores.mean() - cls_baseline_scores.mean():+.4f}"
)


After ProbeSelector ROC-AUC (weighted OVR): 0.9559 ± 0.0016
ΔROC-AUC: -0.0070



## 6. 分類小結
- LightGBM 搭配 SHAP 模式可直接比對真實特徵與 probe 的 shap value baseline，避免缺少 feature_importances_ 的限制。
- `cv='stratified'` 以字串指定，省去手動建立 `StratifiedKFold` 的樣板碼；若日後想改用 group fold，只要改字串即可。
- 迭代 log（`get_iteration_logs()`）能快速列出每輪刪除名單、門檻與 probe 統計，方便調整 `threshold_strategy` 或 `bottom_percent`。



## 7. Regression 範例：Superconductivity Data
接著改用 Superconductivity Data（10k × 81）處理連續目標：建立 `CatBoostRegressor` baseline，再觀察 ProbeSelector 的 collective 模式表現。


In [13]:
SC_SAMPLE_SIZE = 10_000

superconduct = fetch_openml('superconduct', version=1, as_frame=True)
reg_df = superconduct.frame.sample(n=SC_SAMPLE_SIZE, random_state=123)

reg_X = reg_df.drop(columns=['critical_temp']).astype(np.float32)
reg_y = reg_df['critical_temp'].astype(np.float32)
reg_full_df = reg_X.assign(target=reg_y)

cat_reg = CatBoostRegressor(
    depth=8,
    learning_rate=0.05,
    iterations=800,
    loss_function='RMSE',
    random_state=42,
    verbose=0,
    allow_writing_files=False,
)
reg_cv = KFold(n_splits=5, shuffle=True, random_state=42)
reg_scoring = 'neg_root_mean_squared_error'

reg_baseline_scores = cross_val_score(cat_reg, reg_X, reg_y, cv=reg_cv, scoring=reg_scoring)
print(f"Baseline RMSE: {-reg_baseline_scores.mean():.4f} ± {reg_baseline_scores.std():.4f}")
reg_X.head()


Baseline RMSE: 10.1121 ± 0.3139


,number_of_elements,mean_atomic_mass,wtd_mean_atomic_mass,gmean_atomic_mass,wtd_gmean_atomic_mass,entropy_atomic_mass,wtd_entropy_atomic_mass,range_atomic_mass,wtd_range_atomic_mass,std_atomic_mass,...,mean_Valence,wtd_mean_Valence,gmean_Valence,wtd_gmean_Valence,entropy_Valence,wtd_entropy_Valence,range_Valence,wtd_range_Valence,std_Valence,wtd_std_Valence
14288,3.0,20.699179,15.478514,19.210680,14.256599,1.033671,0.962999,16.170538,5.498503,7.076865,...,2.666667,2.730000,2.620741,2.688912,1.082196,0.733966,1.0,1.810000,0.471405,0.443959
10951,1.0,28.085501,28.085501,28.085501,28.085501,0.000000,0.000000,0.000000,0.000000,0.000000,...,4.000000,4.000000,4.000000,4.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000
6480,5.0,120.733879,72.217224,87.791130,42.369816,1.398932,1.312532,184.590607,30.407755,70.963127,...,3.000000,2.059382,2.569470,2.029979,1.430280,1.295651,5.0,0.966746,2.000000,0.541651
8301,6.0,84.188278,98.266968,63.624275,75.158150,1.545959,1.260506,192.981003,50.245174,61.412064,...,2.666667,2.775000,2.492883,2.540489,1.717076,1.480813,3.0,1.175000,1.105542,1.293976
8290,5.0,83.244759,97.046272,59.506081,73.675903,1.311510,1.198595,192.981003,50.245174,67.233833,...,2.600000,2.750000,2.402249,2.514867,1.519383,1.414279,3.0,1.000000,1.200000,1.299038


In [ ]:

reg_selector = ProbeSelector(
    estimator=cat_reg,
    y_col='target',
    feature_cols=None,
    collective=True,
    importance_mode='individual',
    scoring=reg_scoring,
    n_probes=10,
    distribution='all',
    cv='kfold',
    cv_splits=5,
    groups=None,
    random_state=7,
    threshold_strategy='quantile',
    threshold_quantile=0.75,
    iterative=True,
    max_iter=4,
    batch_remove='all_below',
    keep_min_features=25,
    verbose=True,
)

reg_selector.fit(reg_full_df)

print(f"Original features: {reg_X.shape[1]}")
print(f"Selected features: {len(reg_selector.selected_features_)}")
print(f"Probe columns: {len(reg_selector.probe_columns_)} synthetic features")
print(f"Last probe threshold: {reg_selector.last_threshold_:.4f}")
print(f"Dropped features ({len(reg_selector.features_to_drop_)}): {reg_selector.features_to_drop_[:10]}{' ...' if len(reg_selector.features_to_drop_) > 10 else ''}")


[ProbeSelector] Iter 1: 81 -> 63 features (removed 18) | thr=0.4620 | mode=collective
[ProbeSelector] Iter 2: 63 -> 57 features (removed 6) | thr=0.5174 | mode=collective
[ProbeSelector] Iter 3: 57 -> 54 features (removed 3) | thr=0.5167 | mode=collective
[ProbeSelector] Iter 4: 54 -> 54 features (removed 0) | thr=0.4308 | mode=collective
Original features: 81
Selected features: 54
Probe columns: 10 synthetic features
Last probe threshold: 0.4308
Dropped features (27): ['number_of_elements', 'mean_atomic_mass', 'gmean_atomic_mass', 'wtd_gmean_atomic_mass', 'entropy_atomic_mass', 'mean_fie', 'gmean_fie', 'entropy_fie', 'std_fie', 'mean_atomic_radius'] ...


In [15]:

reg_importance = reg_selector.get_importances()
reg_probe_threshold = reg_selector.last_threshold_
reg_iteration_logs = reg_selector.get_iteration_logs()

reg_real_features = (
    reg_importance.loc[~reg_importance['is_probe']]
    .assign(
        threshold=reg_probe_threshold,
        margin=lambda df: df['importance'] - df['threshold'],
        dropped=lambda df: df['feature'].isin(reg_selector.features_to_drop_),
    )
    .sort_values('importance', ascending=False)
)

print('Regression probe importance describe:')
print(reg_importance.loc[reg_importance['is_probe'], 'importance'].describe())

from IPython.display import display

display(reg_real_features.head(10))
display(reg_real_features.query('dropped').sort_values('margin'))
display(reg_selector.get_trace().head(10))
display(reg_iteration_logs)


Regression probe importance describe:
count    10.000000
mean      0.390713
std       0.242299
min       0.036988
25%       0.181280
50%       0.430829
75%       0.584435
max       0.690359
Name: importance, dtype: float64


,feature,importance,is_probe,threshold,margin,dropped
0,range_ThermalConductivity,13.601611,False,0.430829,13.170781,False
1,wtd_gmean_Valence,4.680530,False,0.430829,4.249700,False
2,range_atomic_radius,4.523942,False,0.430829,4.093112,False
3,wtd_mean_Valence,4.062226,False,0.430829,3.631397,False
4,wtd_std_Valence,3.784375,False,0.430829,3.353546,False
5,std_atomic_mass,3.528758,False,0.430829,3.097928,False
6,wtd_mean_ThermalConductivity,3.231645,False,0.430829,2.800816,False
7,wtd_std_ThermalConductivity,2.848650,False,0.430829,2.417821,False
8,wtd_entropy_ThermalConductivity,2.840243,False,0.430829,2.409414,False
9,wtd_gmean_ElectronAffinity,2.736139,False,0.430829,2.305309,False


,feature,importance,is_probe,threshold,margin,dropped


,iteration,feature,importance,threshold,decision,reason,mean,median,min,max,std
0,1,number_of_elements,0.113582,0.461991,remove,below_threshold,0.370661,0.461991,0.045334,0.629477,0.211578
1,1,mean_atomic_mass,0.342846,0.461991,remove,below_threshold,0.370661,0.461991,0.045334,0.629477,0.211578
2,1,wtd_mean_atomic_mass,0.556582,0.461991,keep,above_threshold,0.370661,0.461991,0.045334,0.629477,0.211578
3,1,gmean_atomic_mass,0.411171,0.461991,remove,below_threshold,0.370661,0.461991,0.045334,0.629477,0.211578
4,1,wtd_gmean_atomic_mass,0.400847,0.461991,remove,below_threshold,0.370661,0.461991,0.045334,0.629477,0.211578
5,1,entropy_atomic_mass,0.546839,0.461991,keep,above_threshold,0.370661,0.461991,0.045334,0.629477,0.211578
6,1,wtd_entropy_atomic_mass,1.687030,0.461991,keep,above_threshold,0.370661,0.461991,0.045334,0.629477,0.211578
7,1,range_atomic_mass,0.676630,0.461991,keep,above_threshold,0.370661,0.461991,0.045334,0.629477,0.211578
8,1,wtd_range_atomic_mass,0.839433,0.461991,keep,above_threshold,0.370661,0.461991,0.045334,0.629477,0.211578
9,1,std_atomic_mass,3.469054,0.461991,keep,above_threshold,0.370661,0.461991,0.045334,0.629477,0.211578


,iteration,importance_mode,n_features_start,threshold,n_removed,removed,n_features_after,strategy,probe_mean,probe_median,probe_min,probe_max,probe_std
0,1,collective,81,0.461991,18,"[number_of_elements, mean_atomic_mass, gmean_a...",63,quantile,0.370661,0.461991,0.045334,0.629477,0.211578
1,2,collective,63,0.517418,6,"[std_fie, entropy_atomic_radius, wtd_range_ato...",57,quantile,0.399273,0.517418,0.020421,0.620671,0.234143
2,3,collective,57,0.516710,3,"[entropy_atomic_mass, entropy_fie, mean_Electr...",54,quantile,0.425738,0.516710,0.080965,0.684225,0.228718
3,4,collective,54,0.430829,0,[],54,quantile,0.390713,0.430829,0.036988,0.690359,0.229865



## 8. Regression：重新評估
以同樣的 5-fold KFold + RMSE（`neg_root_mean_squared_error` scorer）驗證 ProbeSelector 在連續目標下的效益。


In [ ]:
reg_selected = reg_selector.transform(reg_full_df)
reg_selected_scores = cross_val_score(cat_reg, reg_selected, reg_y, cv=reg_cv, scoring=reg_scoring)
print(f"After ProbeSelector RMSE: {-reg_selected_scores.mean():.4f} ± {reg_selected_scores.std():.4f}")
print(f"ΔRMSE: {(-reg_selected_scores.mean()) - (-reg_baseline_scores.mean()):+.4f}")


After ProbeSelector RMSE: 10.0850 ± 0.3014
ΔRMSE: -0.0271


In [17]:
reg_selected_scores

array([ -9.93736018, -10.41722577, -10.47805486,  -9.81675617,
        -9.77568221])


## 9. 總結
- Demo 改用更大的 Otto / Superconductivity dataset（皆抽樣 10k），並只使用 LightGBM / CatBoost 這類實務常見模型。
- `importance_mode='shap'` 讓任何可被 SHAP 解釋的模型（如 LightGBM）都能直接利用 probe baseline；若改回 `'collective'` / `'individual'` 也只需換一個參數。
- `cv='stratified'` / `'kfold'` 的字串設計，把交叉驗證的結構含義放進類別本身，不用另外維護 `KFold` 物件。
- `get_iteration_logs()` 提供每輪刪除紀錄，可直接在 notebook 觀察門檻與 probe 統計，方便調整 `threshold_strategy` 或 `bottom_percent`。
